# Day 4 — Attention & Transformers



# 4.1 The Limitation of RNNs

## What is an RNN?

A **Recurrent Neural Network (RNN)** is designed to process sequential data.

For example, consider:

> "The movie was really interesting."

An RNN reads the sentence one word at a time:

```text
The → movie → was → really → interesting
```

At each step, it receives:

1. The current word
2. The hidden state from the previous word

The hidden state acts as a form of memory.

Conceptually:

```text
Word 1
  ↓
Hidden State 1
  ↓
Word 2
  ↓
Hidden State 2
  ↓
Word 3
  ↓
Hidden State 3
```

The problem is that the model has to follow this sequence **step by step**.

---

## The problem with sequential processing

Suppose we have a long sentence:

> "The boy who lived in the small house near the mountain that we visited last summer was extremely happy."

When the model reaches **"was"**, it may need information from **"boy"** much earlier in the sentence.

An RNN has to carry that information through every intermediate step.

```text
boy
 ↓
who
 ↓
lived
 ↓
in
 ↓
the
 ↓
small
 ↓
house
 ↓
...
 ↓
was
```

This creates two major problems.

### 1. Lack of parallelism

An RNN cannot process all words independently during a sequence step because each step depends on the previous hidden state.

This makes training slower.

```text
RNN:

Word 1 → Word 2 → Word 3 → Word 4 → Word 5
          must wait    must wait
```

Transformers change this idea.

```text
Transformer:

Word 1 ─┐
Word 2 ─┤
Word 3 ─┼──→ Attention
Word 4 ─┤
Word 5 ─┘
```

The positions can be processed together during training.

---

## 2. Difficulty with long-range dependencies

RNNs try to preserve information through their hidden state.

For a very long sequence, information may become weaker as it passes through many recurrent steps.

This is related to the **vanishing gradient problem**.

LSTMs improve this problem significantly by introducing mechanisms such as:

* Forget gate
* Input gate
* Output gate
* Cell state

However, an LSTM is still fundamentally sequential.

So LSTM is better than a basic RNN, but it does not completely solve the underlying sequential-processing limitation.

---

## How Transformers solve this

Instead of asking:

> "What information should I carry from the previous step?"

a Transformer can ask:

> **"Which other words in the sequence are important for understanding this word?"**

This is done using **attention**.

---

# 4.2 The Attention Mechanism

## What is Attention?

Attention is a mechanism that allows a model to determine **which parts of the input are most relevant to the current part being processed**.

Consider:

> "The animal didn't cross the road because **it** was tired."

What does **"it"** refer to?

The model needs to understand that "it" most likely refers to **the animal**.

Attention allows the representation of "it" to give more importance to relevant words.

Conceptually:

```text
The     animal     didn't     cross     the     road     because     it
        ↑                                                      ↑
        └──────────────── important relationship ──────────────┘
```

The model doesn't have to remember "animal" through every intermediate step.

It can directly consider the relationship between them.

---

# Self-Attention

The most important idea behind Transformers is **self-attention**.

Self-attention means that the words in a sequence look at **other words in the same sequence** to understand their meaning.

For example:

> "The bank is near the river."

The meaning of **bank** depends on its surrounding words.

Attention allows the model to consider:

```text
bank
 ↓
is
 ↓
near
 ↓
river
```

and determine that "bank" is probably referring to a river bank rather than a financial institution.

---

## Attention scores

The model calculates a score representing how relevant one word is to another.

For example, conceptually:

| Word  | Attention to "bank" |
| ----- | ------------------: |
| The   |                0.02 |
| bank  |                1.00 |
| is    |                0.05 |
| near  |                0.30 |
| the   |                0.02 |
| river |                0.85 |

These numbers are only illustrative.

The important idea is that the model learns **which words should receive more attention**.

---

# Query, Key and Value

The attention mechanism is commonly described using three components:

* **Query (Q)**
* **Key (K)**
* **Value (V)**

You can think about them conceptually like this:

### Query

> "What information am I looking for?"

### Key

> "What kind of information do I contain?"

### Value

> "What information should I actually provide?"

The attention calculation is:

$$
Attention(Q,K,V)
=
softmax
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)V
$$

You don't need to memorize the equation for this training.

The important process is:

```text
Q + K
 ↓
Similarity scores
 ↓
Softmax
 ↓
Attention weights
 ↓
Weighted Values
 ↓
New representation
```

---

## Why divide by √dk?

The dot product between Query and Key vectors can become very large when the vectors have many dimensions.

Large values can make the softmax function extremely concentrated.

Dividing by:

$$
\sqrt{d_k}
$$

helps keep the values at a more reasonable scale and makes training more stable.

---

# Self-Attention Example

Consider:

> "The cat sat on the mat because it was tired."

When processing **"it"**, the model can compare "it" with every other word.

Conceptually:

```text
The       → low relevance
cat       → HIGH relevance
sat       → medium relevance
on        → low relevance
the       → low relevance
mat       → medium relevance
because   → low relevance
it        → current word
was       → medium relevance
tired     → HIGH relevance
```

The model learns these relationships from data rather than us manually defining them.

---

# Why Attention is Powerful

The material highlights three major advantages.

## 1. Self-attention

Every position can consider every other position.

```text
Word 1 ↔ Word 2
Word 1 ↔ Word 3
Word 1 ↔ Word 4
Word 2 ↔ Word 3
Word 2 ↔ Word 4
...
```

This allows the model to capture relationships between distant words.

---

## 2. Parallelism

Unlike an RNN:

```text
RNN:
A → B → C → D → E
```

a Transformer can process the sequence representations together during training:

```text
A ─┐
B ─┤
C ─┼→ Self-Attention
D ─┤
E ─┘
```

This makes Transformer training much more efficient on modern hardware.

**Important:** autoregressive Transformer models such as GPT still generate output tokens one at a time during inference. The major parallelism advantage is especially important during training.

---

## 3. Long-range context

Attention gives a token direct access to other positions.

Instead of:

```text
A → B → C → D → E → F → G
```

information can effectively connect:

```text
A ─────────────────→ G
```

This makes it easier to model relationships between distant parts of text.

---

# Multi-Head Attention

Transformers don't usually use only one attention mechanism.

They use **multiple attention heads**.

For example:

```text
              Input
                │
       ┌────────┼────────┐
       ↓        ↓        ↓
    Head 1    Head 2    Head 3
       ↓        ↓        ↓
     syntax   meaning  context
       └────────┼────────┘
                ↓
           Combined
```

Different heads can learn different relationships.

One head might focus on:

* grammatical relationships

Another might focus on:

* subject/object relationships

Another might focus on:

* contextual relationships

The model learns what each head should focus on.

---

# 4.3 The Transformer Architecture

The original Transformer architecture was introduced in the paper:

> **"Attention Is All You Need" (2017)**

The major change was removing recurrence and building the architecture primarily around attention.

A simplified Transformer looks like:

```text
Input Text
    ↓
Tokenization
    ↓
Token Embeddings
    +
Positional Information
    ↓
Self-Attention
    ↓
Feed-Forward Network
    ↓
More Transformer Layers
    ↓
Output
```

---

# Tokenization

Before text can enter a Transformer, it must be converted into tokens.

For example:

```text
"I love machine learning"
```

could become something similar to:

```text
["I", "love", "machine", "learning"]
```

The tokenizer then converts those tokens into numerical IDs:

```text
[101, 1045, 2293, 3698, 4083, 102]
```

The exact IDs depend on the tokenizer.

The model doesn't directly understand words.

It receives numbers representing tokens.

---

# Embeddings

Token IDs are converted into vectors.

For example:

```text
"cat"
 ↓
[0.21, -0.43, 0.81, ...]
```

These vectors are called **embeddings**.

They allow the neural network to work with mathematical representations of words/tokens.

---

# Positional Encoding

There is an important problem.

Attention itself doesn't naturally understand sequence order.

Consider:

> "Dog bites man."

and:

> "Man bites dog."

The same words are present, but the meaning is completely different.

Therefore, the Transformer needs information about **where each token occurs**.

This is the purpose of positional information.

```text
Token Embedding
      +
Position Information
      ↓
Transformer Input
```

The model can therefore distinguish:

```text
Token A → position 1
Token B → position 2
Token C → position 3
```

from:

```text
Token A → position 3
Token B → position 1
Token C → position 2
```

---

# Feed-Forward Network

After attention, the representation goes through a feed-forward neural network.

A simplified Transformer block is:

```text
Input
  ↓
Self-Attention
  ↓
Add & Normalize
  ↓
Feed-Forward Network
  ↓
Add & Normalize
  ↓
Output
```

This block is repeated multiple times.

More layers allow the model to learn increasingly complex representations.

---

# Encoder vs Decoder

Transformers can be built using different components.

## Encoder

The encoder is mainly used for **understanding input text**.

Examples:

* BERT
* DistilBERT

Useful for:

* Sentiment analysis
* Intent classification
* Text classification
* Semantic analysis
* Named entity recognition

---

## Decoder

The decoder is mainly used for **generating text**.

Examples:

* GPT-2
* GPT-style models

Useful for:

* Text generation
* Chatbots
* Autocomplete
* Generating responses

---

## BERT vs GPT

A simple way to remember the difference:

| Model      | Main purpose                        |
| ---------- | ----------------------------------- |
| BERT       | Understand text                     |
| DistilBERT | Understand text, but smaller/faster |
| GPT-2      | Generate text                       |

For **today's project**, I recommend **DistilBERT** because our first goal is to build a strong understanding/classification component for a chatbot.

---

